# FlashEats - Retrieval and profiling

This notebook retrieves the canonical FlashEats sources, preserves raw Dispatch API pages, proves API completeness, and profiles source grains before modelling.

Canonical data root: `data/raw` in this submission repository.

In [ ]:
from pathlib import Path
import json
import sqlite3
import subprocess
import sys
import time
from datetime import date

import pandas as pd
import requests

ROOT = Path.cwd()
while not (ROOT / 'data' / 'raw' / 'sqlite' / 'flasheats.db').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'raw' / 'sqlite' / 'flasheats.db').exists(), 'Run from the submission repository or a child folder.'
RAW = ROOT / 'data' / 'raw'
FILES = RAW / 'files'
print('Repository:', ROOT)
print('Source files:', len(list(FILES.iterdir())))

/content


AssertionError: Run from the submission repository or a child folder.

## 1. Retrieve local SQL and file sources

The SQLite database is queried directly. CSV and JSON files are loaded without changing the raw copies.

In [ ]:
with sqlite3.connect(RAW / 'sqlite' / 'flasheats.db') as con:
    table_names = pd.read_sql("select name from sqlite_master where type='table' order by name", con)
    sql_sources = {
        table: pd.read_sql(f'select * from {table}', con)
        for table in table_names['name']
    }

file_sources = {}
for path in sorted(FILES.iterdir()):
    if path.suffix == '.csv':
        file_sources[path.stem] = pd.read_csv(path)
    elif path.suffix == '.json':
        file_sources[path.stem] = json.loads(path.read_text(encoding='utf-8'))

source_inventory = []
for name, frame in sql_sources.items():
    source_inventory.append({'source': f'sqlite:{name}', 'grain_hint': 'table', 'rows': len(frame), 'columns': len(frame.columns)})
for name, value in file_sources.items():
    rows = len(value) if isinstance(value, (list, dict)) else None
    columns = len(value.columns) if isinstance(value, pd.DataFrame) else None
    source_inventory.append({'source': f'file:{name}', 'grain_hint': 'file', 'rows': rows, 'columns': columns})

inventory = pd.DataFrame(source_inventory)
display(inventory)

,source,grain_hint,rows,columns
0,sqlite:customers,table,900.0,5.0
1,sqlite:drivers,table,120.0,4.0
2,sqlite:orders,table,1603.0,13.0
3,sqlite:restaurants,table,60.0,6.0
4,file:class7_model_brief,file,4.0,NaN
5,file:client_metric_definitions,file,4.0,NaN
6,file:customer_app_actions,file,NaN,6.0
7,file:customer_interactions,file,NaN,6.0
8,file:driver_events,file,120.0,NaN
9,file:order_events,file,NaN,7.0


## 2. Retrieve the paginated Dispatch API

The mock API intentionally returns transient HTTP 500 and 429 responses. This function retries only transient failures, saves every successful page, and checks the reported total.

In [ ]:
API_URL = 'http://127.0.0.1:8000'
run_date = date.today().isoformat()
raw_api_dir = RAW / 'api' / 'retrieval_runs' / f'run_date={run_date}'
raw_api_dir.mkdir(parents=True, exist_ok=True)

api_process = None
try:
    requests.get(f'{API_URL}/health', timeout=1).raise_for_status()
except requests.RequestException:
    api_process = subprocess.Popen([sys.executable, str(ROOT / 'src' / 'mock_dispatch_api_stdlib.py')], cwd=ROOT / 'src')
    for _ in range(24):
        try:
            if requests.get(f'{API_URL}/health', timeout=1).ok:
                break
        except requests.RequestException:
            time.sleep(0.5)

def fetch_dispatch_pages(page_size=50, max_retries=3):
    session = requests.Session()
    rows = []
    expected_total = None
    page = 1
    while True:
        for attempt in range(1, max_retries + 1):
            response = session.get(f'{API_URL}/dispatch/orders', params={'page': page, 'page_size': page_size}, timeout=5)
            if response.status_code == 200:
                payload = response.json()
                break
            if response.status_code in {429, 500, 502, 503, 504} and attempt < max_retries:
                retry_after = response.headers.get('Retry-After')
                if response.status_code == 429:
                    retry_after = response.json().get('retry_after_seconds', retry_after)
                time.sleep(float(retry_after) if retry_after else 2 ** (attempt - 1))
                continue
            response.raise_for_status()
        if expected_total is None:
            expected_total = int(payload['total_records'])
        (raw_api_dir / f'dispatch_page_{page:03d}.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')
        page_rows = payload.get('data', [])
        rows.extend(page_rows)
        if not payload.get('has_more', False):
            break
        page += 1
    assert len(rows) == expected_total, f'Incomplete Dispatch retrieval: {len(rows)} != {expected_total}'
    assert pd.DataFrame(rows)['order_id'].is_unique, 'Dispatch order grain is not unique'
    return pd.DataFrame(rows), expected_total

dispatch, reported_total = fetch_dispatch_pages()
print({'retrieved': len(dispatch), 'reported_total': reported_total, 'raw_pages': len(list(raw_api_dir.glob('dispatch_page_*.json')))})

if api_process is not None:
    api_process.terminate()

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /dispatch/orders?page=1&page_size=50 (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

## 3. Initial profiling

These checks establish row counts, keys, missingness, and observed categories before any cleaning or modelling.

In [ ]:
orders = sql_sources['orders'].copy()
orders['order_id'].duplicated(keep=False).sum(), orders['order_id'].nunique(), len(orders)

profile = pd.DataFrame({
    'rows': orders.shape[0],
    'unique_orders': orders['order_id'].nunique(),
    'duplicate_rows_involved': orders['order_id'].duplicated(keep=False).sum(),
    'missing_promised_eta': orders['promised_eta'].isna().sum(),
    'missing_actual_delivery_at': orders['actual_delivery_at'].isna().sum(),
}, index=['orders'])
display(profile)
display(orders['final_status'].value_counts(dropna=False).rename_axis('final_status').to_frame('rows'))
display(orders['traffic_bucket'].value_counts(dropna=False).rename_axis('traffic_bucket').to_frame('rows'))